# Cross-Country Climate Comparison
Task 3 template for Ethiopia, Kenya, Sudan, Tanzania, and Nigeria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, kruskal

In [ ]:
countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
frames = []
for c in countries:
    d = pd.read_csv(f'../data/{c}_clean.csv', parse_dates=['DATE'])
    d['Country'] = c.title()
    d['YearMonth'] = d['DATE'].dt.to_period('M').dt.to_timestamp()
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
df.head()

In [ ]:
monthly_t2m = df.groupby(['Country', 'YearMonth'], as_index=False)['T2M'].mean()
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_t2m, x='YearMonth', y='T2M', hue='Country')
plt.title('Monthly Average T2M by Country (2015–2026)')
plt.show()

In [ ]:
t2m_summary = df.groupby('Country')['T2M'].agg(['mean','median','std']).round(3)
precip_summary = df.groupby('Country')['PRECTOTCORR'].agg(['mean','median','std']).round(3)
display(t2m_summary)
display(precip_summary)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Country', y='PRECTOTCORR')
plt.title('PRECTOTCORR Distribution by Country')
plt.show()

In [ ]:
tmp = df.copy()
tmp['Year'] = tmp['DATE'].dt.year
extreme_heat = (tmp.assign(extreme=tmp['T2M_MAX'] > 35)
                .groupby(['Country','Year'], as_index=False)['extreme']
                .sum())
display(extreme_heat.head())

In [ ]:
dry_rows = []
for country, cdf in tmp.sort_values('DATE').groupby('Country'):
    for year, ydf in cdf.groupby('Year'):
        dry = (ydf['PRECTOTCORR'] < 1).astype(int)
        max_streak = streak = 0
        for v in dry:
            if v == 1:
                streak += 1
                max_streak = max(max_streak, streak)
            else:
                streak = 0
        dry_rows.append({'Country': country, 'Year': year, 'max_consecutive_dry_days': max_streak})
dry_days = pd.DataFrame(dry_rows)
display(dry_days.head())

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=extreme_heat, x='Year', y='extreme', hue='Country')
plt.title('Extreme Heat Days (>35°C) by Year')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=dry_days, x='Year', y='max_consecutive_dry_days', hue='Country')
plt.title('Max Consecutive Dry Days (<1mm) by Year')
plt.xticks(rotation=45)
plt.show()

In [ ]:
groups = [g['T2M'].dropna().values for _, g in df.groupby('Country')]
anova_stat, anova_p = f_oneway(*groups)
kruskal_stat, kruskal_p = kruskal(*groups)
print('ANOVA p-value:', anova_p)
print('Kruskal-Wallis p-value:', kruskal_p)

## Vulnerability Ranking
Create a ranking table using:
1. Warming trend strength
2. Precipitation variability (std, IQR)
3. Extreme heat frequency
4. Dry spell length

Then add 5 COP32-framed bullet insights (trend, impact, policy ask).